In [ ]:
import os
import sys
from pathlib import Path
# Add the parent directory to system path to import local modules
sys.path.append(str(Path().absolute().parent))
import torch
import torch.nn.functional as F
import torch.nn.parallel
import torch.utils.data
import torch.utils.data.distributed
import torchvision.transforms as transforms

from GeneratorDataset import GeneratorDataset

from datasets import load_dataset, DownloadConfig, Image
from PIL import Image
import io

from huggingface_hub import login, logout
from tqdm import tqdm

from s3fs import S3FileSystem
import fsspec

import pyarrow
import pyarrow.fs

workers = 0
batch_size = 128
dataset_name = "imagenet-1k"
def arrow_file_generator(shards: list , s3 : S3FileSystem):
    fs_from_s3 = pyarrow.fs.PyFileSystem(pyarrow.fs.FSSpecHandler(s3))
    for file_path in shards:
        try:
            
            with fs_from_s3.open_input_stream(file_path) as stream:
                reader = pyarrow.ipc.open_stream(stream)
                for batch in reader:
                    print(batch.to_pydict())
                    yield batch.to_pydict()
            
        except pyarrow.lib.ArrowInvalid as e:
            print(f"Error reading file {file_path}: {e}")
            continue


def create_dataset_from_arrow_files(shards : list, transform : callable, s3 : S3FileSystem):

    generator_object = arrow_file_generator(shards=shards,s3=s3)
    created_dataset = GeneratorDataset(
    generator=generator_object,
    transform=transform
    )
    return created_dataset

class Basic_Dataloader():
    def __init__(self, local_dataset,split):
        self.train_dataset = None
        self.train_loader = None
        self.local_dataset = local_dataset
        self.split = split
        return None


    @staticmethod
    def convert_bytes_to_image(input_image_handle):
        print(type(input_image_handle))
        try:
            image = Image.open(io.BytesIO(input_image_handle))
            return image
        except Exception as e:
            try:
                if isinstance(input_image_handle,Image.Image):
                    return input_image_handle
                else:
                    print("input is not an image or byte array")
                    raise TypeError
            except Exception as e:
                print(f"Error converting bytes to image: {e}")
                return None
        
    @staticmethod
    def train_transforms(examples):
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        nn_transforms =  transforms.Compose(
            [
                transforms.RandomResizedCrop(224), 
                transforms.RandomHorizontalFlip(), 
                transforms.ToTensor(), 
                normalize
            ]

        )

        try:
            examples["pixel_values"] = [
            nn_transforms(
                Basic_Dataloader.convert_bytes_to_image(image_data["bytes"]).convert("RGB")
            )
                for image_data in examples["image"]
            ]
            return examples
        except Exception as e:
            try:
                examples["pixel_values"] = [
                nn_transforms(
                    Basic_Dataloader.convert_bytes_to_image(examples["image"]).convert("RGB")
                )
                ]
                return examples
            except Exception as e:
                print(e)
                return None

    @staticmethod
    def collate_fn(batch):
        #print(type(batch))
        #print(batch.keys())
        #print(batch[0].keys())
        if batch:
            return {
                'pixel_values': torch.stack([x['pixel_values'][0] for x in batch]),
                'label': torch.tensor([x['label'] for x in batch])
            }
        else:
            return None
    def worker_init_fn(self,worker_id):
        if self.local_dataset == True:
            self.train_dataset.worker_initializer(
                    Basic_Dataloader.train_transforms,
                    self.split,
                    )
    def train_dataloader(self):
        train_dataset = None

        if self.local_dataset == True:

            #access_key =os.environ["MINIO_ACCESS_KEY"]
            #secret_key = os.environ["MINIO_SECRET_KEY"]
            #url = "https://10.0.1.86:443"

            

            #fs = S3FileSystem(
            #    key = access_key,
            #    secret=secret_key,
            #    use_ssl=True,
            ##    endpoint_url=url,
            #    client_kwargs={"verify":False}
            #)            


            #s3_storage_options = {
            #    "protocol" : "s3",
            #    "key" : access_key,
            #    "secret" : secret_key,
            #    "use_ssl" : True,
            #    "endpoint_url" : url,
            #    "client_kwargs" : {"verify" : False},
            #}

            #dataset_path = ""
            #if self.split == 'train':
            #    dataset_path="hugging-face-bucket-imagenet-1k"
            #elif self.split == "test":
            #    dataset_path="hugging-face-bucket-imagenet-1k-test"
            #else:
            #    dataset_path="hugging-face-bucket-imagenet-1k-validation"
            #S3_URI = f"s3://{dataset_path}"
            #test_fs : S3FileSystem
            #test_fs, path = fsspec.url_to_fs(S3_URI,**s3_storage_options)
            #all_files = test_fs.ls(S3_URI)
            #arrow_files = [file for file in all_files if file.endswith('.arrow')]
            #self.train_dataset = create_dataset_from_arrow_files(
            #    arrow_files,
            #    Basic_Dataloader.train_transforms,
            #    test_fs
            #    )
            self.train_dataset = GeneratorDataset()
            self.train_dataset.worker_initializer(
                Basic_Dataloader.train_transforms,
                self.split
            )


        else:

            d_cfg = DownloadConfig(
            resume_download = True,
            max_retries= 500
            )

            login(
                token=os.environ['HF_TOKEN']
            )

            self.train_dataset = load_dataset(
                dataset_name, 
                split='train', 
                streaming=True, 
                token=os.environ['HF_TOKEN'], 
                trust_remote_code=True,
                download_config=d_cfg,
                )
            
            logout()
            self.train_dataset = self.train_dataset.map(Basic_Dataloader.train_transforms)
            self.train_dataset = self.train_dataset.with_format("torch")  
        
        #print(self.train_dataset)
            
        #train_dataset = split_dataset_by_node(train_dataset, rank=self.rank,world_size=self.world_size)


        try:
            #self.train_dataset = train_dataset
            if self.local_dataset == True:
                self.train_loader = torch.utils.data.DataLoader(
                    dataset=self.train_dataset, 
                    batch_size=batch_size, 
                    shuffle=None, 
                    num_workers=workers,
                    collate_fn=Basic_Dataloader.collate_fn,      
                    #pin_memory=True,
                    drop_last=True,    
                    worker_init_fn=self.worker_init_fn,  
                )
            else:
                self.train_loader = torch.utils.data.DataLoader(
                    dataset=self.train_dataset, 
                    batch_size=batch_size, 
                    shuffle=None, 
                    num_workers=workers,
                    collate_fn=Basic_Dataloader.collate_fn,      
                    #pin_memory=True,
                    drop_last=True,      
                )
        except Exception as exc:
            raise exc

        return self.train_loader

bdl = Basic_Dataloader(local_dataset=False,split='train')
data_loader = bdl.train_dataloader()
for epoch in range(2):
    #print(data_loader)
    data_iterator = tqdm(
        data_loader,
        leave=True,
        unit="batch",
        postfix={
            "step": "training",
            "epo": 0,
            "lss": "%.8f" % 0.0,
            "acc1": "%.4f" % 0.0,
            "acc5": "%.4f" % 0.0,
        },
        disable=False,
        )

    for batch_idx, batch in enumerate(data_iterator):
    #for batch in next(iter(data_iterator)):
        batch_b = batch
        #if batch is not None:
        #    print(batch_idx)
        #    batch = None
        print(f"batch: {batch}")
        #print(f"batch_idx: {batch_idx}")
        #break
        